<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 5: *Subset and Split*
##### Version Number: 4.0
---
### Contents
> *One Hot Encoding*\
> *Separate Dates for modeling*\
> *Downsize Data*\
> *Split Data*\
> *Export Data*
---
### Notes
---
### Inputs
- engineered_samples.csv - Complete main dataset
---
### Outputs 
- `X_ignition`,`X_damage`, `X_spread`, `pal_X` - numeric and bool columns for modeling
- `y_ignition`,`y_spread`,`y_damage` - target data
- `ignition_details`,`spread_details`,`damage_details`,`pal_details` - text columns and spatial info
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

from rich.console import Console
from rich.table import Table
from rich.columns import Columns

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

---

###  Load Data

In [3]:
samples = pd.read_csv("../data/processed/engineered_samples.csv")

In [4]:
samples['State Region'] = 'Southern'
samples.loc[samples['centroid_northing'] > -113000,'State Region'] = 'Northern'

In [5]:
samples['State Region'].value_counts()

State Region
Southern    324953
Northern    283690
Name: count, dtype: int64

In [6]:
console = Console()

## Original Full Set Distributions

In [7]:
display_values(samples['Target_Ignition'],
    samples['Target_Spread'],
    samples['Target_Damage'],
              console)

  Ignition Value      Spread Value       Damage Value   
      Counts             Counts             Counts      
┏━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━┳━━━━━━━━┓
┃ Index ┃  Value ┃ ┃ Index ┃  Value ┃ ┃ Index ┃  Value ┃
┡━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━╇━━━━━━━━┩
│ 0     │ 565658 │ │ 0     │ 587380 │ │ 0     │ 601686 │
│ 1     │  42985 │ │ 1     │   5349 │ │ 2     │   1760 │
└───────┴────────┘ │ 3     │   5332 │ │ 1     │   1744 │
                   │ 4     │   5298 │ │ 4     │   1736 │
                   │ 2     │   5284 │ │ 3     │   1717 │
                   └───────┴────────┘ └───────┴────────┘

## One Hot Encoding

One hot encode appropriate categorical variables.

In [8]:
categorical_columns = ['dominant_province_description','dominant_section_description','Season', 'State Region']
one_hot = samples[categorical_columns]
samples = samples.drop(columns = ['dominant_province_description','dominant_section_description','Season', 'State Region'])

one_hot_samples = pd.get_dummies(
    one_hot,
    columns=categorical_columns,
    drop_first=False
).astype(int)

encoded_samples = pd.concat([samples,one_hot_samples],axis=1)

### Separate Dates for modeling and case study

- **01/01/2018 - 12/31/2024** Dates to train the models
- **01/01/2025 - 01/23/2025** Dates for evaluating model performance on unseen data

In [9]:
encoded_samples['Date'] = pd.to_datetime(encoded_samples['Date'])

# first day to analyze in weather dataset
FIRST_DATE = pd.to_datetime('2018-02-01').date()

# last day to analyze in weather dataset
LAST_DATE = pd.to_datetime('2024-12-31').date()

# Boolean mask for dates within the range
mask = (encoded_samples['Date'].dt.date >= FIRST_DATE) & (encoded_samples['Date'].dt.date <= LAST_DATE)

# Keep only rows within the range
model_samples = encoded_samples.loc[mask].copy()

In [10]:
mask = (encoded_samples['Date'].dt.date >= pd.to_datetime('2025-01-01').date()) & (encoded_samples['Date'].dt.date <= pd.to_datetime('2025-01-22').date())
pal = encoded_samples.loc[mask].copy()

### (OPTIONAL) Subset classes to save processor
- Downsize the majority class in all three sets

In [11]:
model_samples['Date'] = pd.to_datetime(model_samples['Date'])

northern = model_samples[model_samples['State Region_Northern'] == 1]
southern = model_samples[model_samples['State Region_Southern'] == 1]

ignition_reduced_northern = subset_df_cap_majority(northern,'Target_Ignition',20000)
spread_reduced_northern = subset_df_cap_majority(northern,'Target_Spread',20000)
damage_reduced_northern = subset_df_cap_majority(northern,'Target_Damage',20000)

ignition_reduced_southern = subset_df_cap_majority(southern,'Target_Ignition',20000)
spread_reduced_southern = subset_df_cap_majority(southern,'Target_Spread',20000)
damage_reduced_southern = subset_df_cap_majority(southern,'Target_Damage',20000)

In [12]:

ignition_reduced = pd.concat([ignition_reduced_northern, ignition_reduced_southern],
    ignore_index=True)

spread_reduced = pd.concat([spread_reduced_northern, spread_reduced_southern],
    ignore_index=True)

damage_reduced = pd.concat([damage_reduced_northern, damage_reduced_southern],
    ignore_index=True)

In [13]:
display_values(ignition_reduced['Target_Ignition'],
    spread_reduced['Target_Spread'],
    damage_reduced['Target_Damage'],
              console)

 Ignition Value     Spread Value      Damage Value   
     Counts            Counts            Counts      
┏━━━━━━━┳━━━━━━━┓ ┏━━━━━━━┳━━━━━━━┓ ┏━━━━━━━┳━━━━━━━┓
┃ Index ┃ Value ┃ ┃ Index ┃ Value ┃ ┃ Index ┃ Value ┃
┡━━━━━━━╇━━━━━━━┩ ┡━━━━━━━╇━━━━━━━┩ ┡━━━━━━━╇━━━━━━━┩
│ 0     │ 40000 │ │ 0     │ 40000 │ │ 0     │ 40000 │
│ 1     │ 37365 │ │ 1     │  5325 │ │ 2     │  1760 │
└───────┴───────┘ │ 3     │  5291 │ │ 1     │  1744 │
                  │ 2     │  5281 │ │ 3     │  1700 │
                  │ 4     │  5265 │ │ 4     │  1683 │
                  └───────┴───────┘ └───────┴───────┘

## Split Data

In [14]:
text_columns = ['Sample_ID', 'Date', 'grid_id',
       'geometry','area_in_cali','maximum_x', 'minimum_y',
       'maximum_y', 'minimum_x','centroid_northing','centroid_easting','Target_Damage',
                'Target_Ignition', 'Target_Spread','Year','acres','total_fire_damage','damage_per_day',
               'acres_per_day','damage_so_far','acres_burned_so_far']

## Modeling Data
y_ignition = ignition_reduced['Target_Ignition']
y_spread = spread_reduced['Target_Spread']
y_damage = damage_reduced['Target_Damage']

X_ignition = ignition_reduced.drop(columns=text_columns)
X_spread = spread_reduced.drop(columns=text_columns)
X_damage = damage_reduced.drop(columns=text_columns)

details_ignition = ignition_reduced[text_columns]
details_spread = spread_reduced[text_columns]
details_damage = damage_reduced[text_columns]

## Case study data
pal_X = pal.drop(columns=text_columns)
pal_y = pal[['Target_Ignition','Target_Spread','Target_Damage']]
pal_details = pal[text_columns]


## Export Data

In [15]:
X_ignition.to_csv('../data/processed/X_ignition.csv', index=False)
X_spread.to_csv('../data/processed/X_spread.csv', index=False)
X_damage.to_csv('../data/processed/X_damage.csv', index=False)

y_ignition.to_csv('../data/processed/y_ignition.csv', index=False)
y_spread.to_csv('../data/processed/y_spread.csv', index=False)
y_damage.to_csv('../data/processed/y_damage.csv', index=False)

details_ignition.to_csv('../data/processed/details_ignition.csv', index=False)
details_spread.to_csv('../data/processed/details_spread.csv', index=False)
details_damage.to_csv('../data/processed/details_damage.csv', index=False)

pal_X.to_csv('../data/processed/pal_X.csv', index=False)
pal_y.to_csv('../data/processed/pal_y.csv', index=False)
pal_details.to_csv('../data/processed/pal_details.csv', index=False)

print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
